In [6]:
import os
import sys
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)



from Frame import Frame
import Plotters
import Utils as Utils
import matplotlib.pyplot as plt
import numpy as np
from Frame import Frame
import Utils
import pickle

import matplotlib.pyplot as plt
import Plotters
import plotly.graph_objects as go
from plyfile import PlyData

import pickle
import numpy as np

import matplotlib.pyplot as plt
import Utils
%matplotlib qt


path = 'C:/Users/Roni/Documents/gs_input/frames_model.pkl'


# dict_path = 'D:/Documents/data_for_gs/fly_gray/dict/frames_model.pkl'







path_output = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/hull_output'
path_output = 'D:/Documents/gaussian_model_output/'

dict_path  = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/frames_model.pkl'
image_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/'
model_name = 'fly_to_bee'
file_name = 'bee_model_dense_10000'


# dict_path  = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/frames_model.pkl'
# image_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/'
# model_name = 'only_fly'
# file_name = 'fly_model'

model_name = 'fly_to_fly'
file_name = 'fly_model'

# dict_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/frames_model.pkl'
# image_path =  'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/'
# model_name = '3dgs_bee'
# file_name = 'bee_try'
path_output = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/'
path_output = 'D:/Documents/gaussian_model_output/'

# dict_path  = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/frames_model.pkl'
# image_path = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/'
# model_name = 'fly_to_bee'
# file_name = 'bee_model_dense_10000'

model_name = 'fly_fast'
file_name = 'fly_model_fast'

dict_path  = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/frames_model.pkl'
image_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/'

model_name = 'fly_evaluation'
file_name = 'fly_model_5000_itr'
dict_path  = 'D:/Documents/data_for_gs/mov1_2023_08_09_60ms/dict/frames_model.pkl'
image_path = 'D:/Documents/data_for_gs/mov1_2023_08_09_60ms/'


path_angles = f'{path_output}/{model_name}/{file_name}_angles.pkl'
path_results = f'{path_output}/{model_name}/{file_name}_angles.pkl'

h5_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evalutation'

# download model_run localy
path = f'D:/Documents/gaussian_model_output/{model_name}/{file_name}.pkl'
if os.path.exists(f'{path}'):
    with open(path, 'rb') as handle:
        output_angles_weights = pickle.load(handle)

iteration = 1200

frame0 = 370
frame_end = 390
weight_flag = False

with open(dict_path,'rb') as f:
    frames = pickle.load(f)

vertices_list = []
image_list = []
weights_list = []
gaussian_list = []
idx_parts = []
xyz_rotated = []
for frame in range(frame0,frame_end):
    ew_to_lab = frames[frame][1][list(frames[frame][1].keys())[0]]['ew_to_lab']
    input_dir = f'{path_output}/{model_name}/{file_name}'
    input_file = f'{path_output}/{model_name}/{frame}/{file_name}/point_cloud/iteration_{iteration}/point_cloud.ply'
    vertices = PlyData.read(input_file)["vertex"]
    xyz = np.column_stack((vertices['x'],vertices['y'],vertices['z']))
    vertices_list.append(xyz)
    xyz_rotated.append((ew_to_lab @ xyz.T).T)

    frames_per_cam = [Frame(image_path,frame,cam, frames_dict = frames)  for cam in range(4)]
    [frame.interest_point_crop(h5_path,frame0 = 370) for frame in frames_per_cam]
    image_list.append(frames_per_cam)
    if os.path.exists(f'{input_dir}_results.pkl'):
        with open(f'{input_dir}_results.pkl', 'rb') as handle:
            output_angles_weights = pickle.load(handle)
        weights_list.append(output_angles_weights['weights'])
        weight_flag = True
        idx_parts.append([np.sum(output_angles_weights['weights'][frame - frame0][iteration][:,idx:idx + 3],axis = 1) == 1 for idx in range(0,9,3)])
        color_list = ['lime','crimson','dodgerblue']
        color_list_2d = ['lime','crimson','dodgerblue']





# frames_per_cam = [Frame(image_path,frame,cam_num, frames_dict = frames) for cam_num in range(4)]


In [7]:
frame = 385



point_3d_per_frame = []
gaussians_interest_points = []
for frame in range(frame0,frame_end):
    ew_to_lab = frames[frame][1][list(frames[frame][1].keys())[0]]['ew_to_lab']
    image_list[frame - frame0][0].interest_points[14,:] = 999
    image_list[frame - frame0][1].interest_points[14,:] = 999
    # image_list[frame - frame0][2].interest_points[14,:] = 999
    # image_list[frame - frame0][3].interest_points[14,:] = 999
    points_3d = []
    for idx in range(len(image_list[frame - frame0][0].interest_points)):
        center_pixel_dir = []
        camera_location = []
        for frame_obj in image_list[frame - frame0]:
            if (frame_obj.interest_points[idx] !=999).all():
                center_pixel_dir.append(frame_obj.camera_center_to_pixel_ray(frame_obj.interest_points[idx]))
                camera_location.append(frame_obj.X0)
        
        # point = np.array([interest_point[0],interest_point[1]])
        # center_pixel_dir = [frame.camera_center_to_pixel_ray(frame.interest_points[idx]) for frame in image_list[frame_number - frame0] if (frame.interest_points[idx] !=999).all() ]
        # camera_location = [frame.X0 for frame in image_list[frame_number - frame0] if (frame.interest_points[idx] !=999).all() ]
        # cm_pix = [frame.camera_center_to_pixel_ray(frame.cm) for frame in image_list[frame_number - frame0]]
        points_3d.append(Utils.triangulate_least_square(np.hstack(camera_location).T,np.vstack(center_pixel_dir)))
        if idx == 14:
            print(center_pixel_dir)
            idx_14_cam = camera_location
            idx_14_pix = center_pixel_dir

    rotated_points_3d = (ew_to_lab @ np.vstack(points_3d).T).T
    sorted_dist = [np.argsort(np.sqrt(np.sum((xyz_rotated[frame - frame0]- point)**2, axis = 1)))[0:1] for point in rotated_points_3d]
    gaussian_points = np.vstack([np.mean(xyz_rotated[frame - frame0][sorted_idx,:], axis = 0) for sorted_idx in sorted_dist])


    point_3d_per_frame.append(np.vstack(points_3d))
    gaussians_interest_points.append(gaussian_points)

[array([[-0.56773942, -0.55661401, -0.07030329]]), array([[ 0.03936628,  0.09735938, -0.80048359]])]
[array([[-0.56829362, -0.55605803, -0.07025791]]), array([[ 0.03863654,  0.09777143, -0.8004641 ]])]
[array([[-0.5684716 , -0.55586428, -0.07052036]]), array([[ 0.03820802,  0.09770405, -0.80046988]])]
[array([[-0.56884432, -0.55548574, -0.07057409]]), array([[ 0.0379302 ,  0.09787188, -0.80046185]])]
[array([[-0.56912858, -0.55519405, -0.07066958]]), array([[ 0.03759965,  0.09798623, -0.80045704]])]
[array([[-0.56914816, -0.55516022, -0.07092654]]), array([[ 0.03746769,  0.09794381, -0.80046003]])]
[array([[-0.56947482, -0.55482491, -0.07103818]]), array([[ 0.03694835,  0.09799641, -0.80045956]])]
[array([[-0.56952763, -0.55475353, -0.07136916]]), array([[ 0.03685809,  0.09799844, -0.80045987]])]
[array([[-0.57011099, -0.55415614, -0.0715429 ]]), array([[ 0.03618356,  0.09840718, -0.8004403 ]])]
[array([[-0.5700608 , -0.55420715, -0.07153508]]), array([[ 0.03595743,  0.09809391, -0.800

In [8]:
point_3d_per_frame

[array([[ 0.01058323,  0.01269727, -0.00875624],
        [ 0.00962417,  0.01328644, -0.00938736],
        [ 0.00947689,  0.0138137 , -0.00956724],
        [ 0.009647  ,  0.01397909, -0.00947368],
        [ 0.01042519,  0.01389401, -0.00889915],
        [ 0.01099652,  0.01314689, -0.00843779],
        [ 0.0103713 ,  0.01332453, -0.00897037],
        [ 0.01102142,  0.0123529 , -0.00822422],
        [ 0.01080521,  0.01105315, -0.00734799],
        [ 0.01012197,  0.01018857, -0.00662025],
        [ 0.010119  ,  0.01003657, -0.00617375],
        [ 0.01030779,  0.01017955, -0.00605093],
        [ 0.01092343,  0.01084327, -0.00622154],
        [ 0.01125766,  0.01148774, -0.00706648],
        [ 0.01083059,  0.01078379, -0.00674772],
        [ 0.01110115,  0.01172403, -0.00767082],
        [ 0.01212884,  0.0129923 , -0.00711996],
        [ 0.01009439,  0.01164068, -0.00824943]]),
 array([[ 0.0105458 ,  0.01267232, -0.00875224],
        [ 0.00953772,  0.01320181, -0.00935996],
        [ 0.009344

In [3]:


# input_file = f'D:/Documents/gaussian_model_output/mdoel_le_deform/{frame}/{file_name}/point_cloud/iteration_900/point_cloud.ply'


frame_data = image_list[frame - frame0] 


ew_to_lab = frames[frame][1][list(frames[frame][1].keys())[0]]['ew_to_lab']
xyz = vertices_list[frame - frame0]
# xyz_rotated = (ew_to_lab @ xyz.T).T


if weight_flag == False:
    idx_parts = list(range(len(xyz)))
    color_list_2d = ['red']

fig = go.Figure()

for idx,color in zip(range(len(idx_parts[frame - frame0])),color_list):   
    Plotters.scatter3d(fig,xyz_rotated[frame - frame0][idx_parts[frame - frame0][idx],:],color,3,'body',show_colorbar = False)

rotated_points_3d = (ew_to_lab @ np.vstack(points_3d).T).T
sorted_dist = [np.argsort(np.sqrt(np.sum((xyz_rotated[frame - frame0]- point)**2, axis = 1)))[0:1] for point in rotated_points_3d]
gaussian_points = np.vstack([np.mean(xyz_rotated[frame - frame0][sorted_idx,:], axis = 0) for sorted_idx in sorted_dist])
gaussian_points_ew = (ew_to_lab.T @ np.vstack(gaussian_points).T).T

Plotters.scatter3d(fig,rotated_points_3d,'black',5,'interest',show_colorbar = False)
Plotters.scatter3d(fig,gaussian_points,'orange',5,'gaussians',show_colorbar = False)



fig.show()
fig.write_html(f'{path_output}/plot_hull.html')

ax = None
for idx,color in zip(range(len(idx_parts[frame - frame0])),color_list_2d): 
    ax = Plotters.plot_projections(xyz[idx_parts[frame - frame0][idx],:],frame_data,color = color,ax = ax)

ax = Plotters.plot_projections(np.vstack(points_3d),frame_data,color = 'magenta',ax = ax, size = 5)
ax = Plotters.plot_projections(gaussian_points_ew,frame_data,color = 'orange',ax = ax, size = 5)

ax = None
ax = Plotters.plot_projections(np.vstack(points_3d),frame_data,color = 'magenta',ax = ax, size = 5)
ax = Plotters.plot_projections(gaussian_points_ew,frame_data,color = 'orange',ax = ax, size = 5)

for idx in range(4):
    points2d = frames_per_cam[idx].project_with_proj_mat(gaussian_points_ew)[:,0:2]
    np.sqrt(np.sum((points2d - image_list[frame - frame0][idx].interest_points)**2,axis = 1))


for idx in range(4):
    points = image_list[frame - frame0][idx].interest_points
    ax[idx//2,np.mod(idx,2)].scatter(points[:,1] ,points[:,0] ,color = 'white', alpha = 1, s= 5,cmap = 'white')

rotated_points_3d[14]

d:\Documents\model_gaussian_splatting\Plotters.py:71: UserWarning:

No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored

C:\Users\Roni\AppData\Local\Temp\ipykernel_4708\3111310755.py:52: UserWarning:

No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored



array([ 0.00998357, -0.01261418, -0.00373519])

In [9]:
import plotly.graph_objects as go
import numpy as np

# === CONFIG ===
frames = range(frame0, frame_end)
output_path = f'{path_output}/{model_name}/animated_plot.html'


# === HELPERS ===

def create_scatter3d(xyz, idx_parts, color):
    """Create a single 3D scatter trace for a specific part."""
    return go.Scatter3d(
        x=xyz[idx_parts, 0],
        y=xyz[idx_parts, 1],
        z=xyz[idx_parts, 2],
        mode="markers",
        marker=dict(size=2, opacity=1, color=color, colorscale='gray'),
    )


def get_global_bounds(xyz_list):
    """Compute global min and max coordinates over all frames for consistent axis scaling."""
    stacked = np.vstack(xyz_list)
    return np.min(stacked, axis=0), np.max(stacked, axis=0)


def create_frame(xyz_frame, idx_parts_list, color_list, frame_name):
    """Create one animation frame with all parts for a given timestep."""
    data = [
        create_scatter3d(xyz_frame, idx_parts, color)
        for idx_parts, color in zip(idx_parts_list, color_list)
    ]
    return go.Frame(data=data, name=frame_name)


def create_play_pause_buttons():
    """Return Play/Pause button definitions for animation."""
    return [
        {
            "buttons": [
                {
                    "args": [None, {"frame": {"duration": 100, "redraw": True}, "fromcurrent": True}],
                    "label": "Play",
                    "method": "animate",
                },
                {
                    "args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                    "label": "Pause",
                    "method": "animate",
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 87},
            "showactive": False,
            "type": "buttons",
            "x": 0.1,
            "xanchor": "right",
            "y": 0,
            "yanchor": "top",
        }
    ]


def create_slider(frames_range):
    """Create a slider object to control the animation."""
    return [
        {
            "active": 0,
            "steps": [
                {
                    "args": [[str(i)], {"frame": {"duration": 100, "redraw": True}, "mode": "immediate"}],
                    "label": str(frame),
                    "method": "animate",
                }
                for i, frame in enumerate(frames_range)
            ],
        }
    ]


# === MAIN FUNCTION ===

def create_3d_animation(xyz_all_frames, idx_parts_list, color_list):
    """Build and show the 3D animation."""
    min_xyz, max_xyz = get_global_bounds(xyz_all_frames)

    # Initial frame data
    initial_data = [
        create_scatter3d(xyz_all_frames[0], idx_parts, color)
        for idx_parts, color in zip(idx_parts_list[0], color_list)
    ]
    bounding_box_trace = go.Scatter3d(
    x=[min_xyz[0], max_xyz[0]],
    y=[min_xyz[1], max_xyz[1]],
    z=[min_xyz[2], max_xyz[2]],
    mode='markers',
    marker=dict(size=0.1, color='rgba(0,0,0,0)'),
    showlegend=False
    )
    initial_data.append(bounding_box_trace)
    # Create frames for animation
    frames_data = [
        create_frame(xyz_frame, idx_parts_list[i], color_list, str(i))
        for i, xyz_frame in enumerate(xyz_all_frames)
    ]



    # Build full figure
    fig = go.Figure(
        data=initial_data,
        layout=go.Layout(
            scene=dict(
                xaxis_title="X",
                yaxis_title="Y",
                zaxis_title="Z",
            ),
            updatemenus=create_play_pause_buttons(),
            sliders=create_slider(frames),
        ),
        frames=frames_data,
    )

    fig.show()
    fig.write_html(output_path)
    print(f"Saved animation to: {output_path}")


create_3d_animation(xyz_rotated, idx_parts, color_list)




    # point_3d_per_frame.append(np.vstack(points_3d))
    # gaussians_interest_points.append(gaussian_points)

IndexError: boolean index did not match indexed array along dimension 0; dimension is 3690 but corresponding boolean dimension is 3740

In [255]:
h5_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evalutation'
[frame.interest_point_crop(h5_path,frame0 = 370) for frame in image_list[frame_number - frame0]]
points_3d = []
for idx in range(len(image_list[frame_number - frame0][0].interest_points)):
    # point = np.array([interest_point[0],interest_point[1]])
    center_pixel_dir = [frame.camera_center_to_pixel_ray(frame.interest_points[idx]) for frame in image_list[frame_number - frame0]]
    camera_location = [frame.X0 for frame in image_list[frame_number - frame0]]
    cm_pix = [frame.camera_center_to_pixel_ray(frame.cm) for frame in image_list[frame_number - frame0]]
    points_3d.append(Utils.triangulate_least_square(np.hstack(camera_location).T,np.vstack(center_pixel_dir)))


In [256]:
dist_from_interest_point = []
for frame_number in range(370,470):
    [image_list[frame_number - frame0][idx].interest_point_crop(h5_path,frame0 = 370) for idx in range(4)]
    points_3d = []
    for idx in range(len(image_list[frame_number - frame0][0].interest_points)):
        # point = np.array([interest_point[0],interest_point[1]])
        center_pixel_dir = [frame.camera_center_to_pixel_ray(frame.interest_points[idx]) for frame in image_list[frame_number - frame0]]
        camera_location = [frame.X0 for frame in image_list[frame_number - frame0]]
        points_3d.append(Utils.triangulate_least_square(np.hstack(camera_location).T,np.vstack(center_pixel_dir)))

    rotated_points_3d = (ew_to_lab @ np.vstack(points_3d).T).T
    sorted_dist = [np.argsort(np.sqrt(np.sum((xyz_rotated[frame_number - frame0]- point)**2, axis = 1)))[0:1] for point in rotated_points_3d]
    gaussian_points = np.vstack([np.mean(xyz_rotated[frame_number - frame0][sorted_idx,:], axis = 0) for sorted_idx in sorted_dist])
    gaussian_points_ew = (ew_to_lab.T @ np.vstack(gaussian_points).T).T

    for idx_cam in range(4):
        points2d = image_list[frame_number - frame0][idx_cam].project_with_proj_mat(gaussian_points_ew)[:,0:2]
        dist_from_interest_point.append(np.sqrt(np.sum((np.fliplr(points2d) - image_list[frame_number - frame0][idx_cam].interest_points)**2,axis = 1)))



In [258]:
plt.figure()
plt.hist(np.vstack(dist_from_interest_point)[:,14], stacked=True, bins = 20)
plt.show()

In [145]:
dist_from_interest_point

[array([1.1446717 , 0.73141199, 1.35537464, 0.86834903, 1.01204996,
        0.79041449, 1.25712847, 0.22199912, 1.23795535, 0.43945316,
        0.51018276, 0.32948951, 0.73816476, 0.86528359, 1.77954381,
        1.02944227, 0.68216725, 1.82579696]),
 array([1.42192617, 0.59346322, 1.06731211, 1.02990715, 3.05189129,
        0.73650216, 1.35127659, 0.26017687, 1.17455996, 0.2770593 ,
        0.51691408, 0.09602975, 1.69254191, 1.55883884, 1.16907255,
        1.60212554, 0.77803327, 0.82516979]),
 array([0.60686234, 0.52946272, 0.33976691, 1.03629318, 2.81814332,
        1.83095703, 1.27039671, 1.29724788, 0.8401179 , 0.90112702,
        1.67847275, 0.7623813 , 1.89750297, 1.87480317, 3.24094596,
        1.84101734, 0.8268814 , 1.67088057]),
 array([0.93019241, 0.61618048, 1.0363388 , 0.99356129, 1.77392801,
        0.45771845, 0.58309018, 0.68283064, 0.29328219, 0.44478091,
        0.55497287, 0.75242974, 0.97682981, 0.49578432, 1.69402358,
        0.67307767, 0.58840987, 1.81556397]),


In [154]:
dist_from_interest_point = []
for idx in range(4):
    points2d = image_list[frame_number - frame0][idx].project_with_proj_mat(gaussian_points_ew)[:,0:2]
    dist_from_interest_point.append(np.sqrt(np.sum((np.fliplr(points2d) - image_list[frame_number - frame0][idx].interest_points)**2,axis = 1)))
dist_from_interest_point

[array([1.06488346, 1.6456344 , 0.9191633 , 0.34555581, 0.47133398,
        1.41421306, 1.66563396, 0.77138527, 0.83902014, 0.26940481,
        0.41232041, 0.10392812, 1.17198434, 0.58292617, 1.855086  ,
        0.72436482, 0.52537062, 1.68445405]),
 array([0.72215088, 1.78376872, 0.52655123, 0.42996084, 0.96870301,
        1.68707388, 2.67764813, 0.8055551 , 0.71439518, 1.70473045,
        0.96765147, 0.52274736, 1.8028318 , 0.77090183, 1.92662156,
        0.82632004, 0.29744224, 0.55337854]),
 array([0.35887793, 0.27036293, 1.4422276 , 0.98677969, 1.28996097,
        0.47950741, 0.0139633 , 0.72659842, 0.86376551, 2.49544906,
        1.84803195, 1.10297277, 1.30291477, 0.99268519, 1.31925614,
        2.01583895, 0.62242227, 1.90630242]),
 array([1.07637437, 1.12175682, 1.6982961 , 0.87757487, 0.81625889,
        0.68561221, 3.16273123, 0.83478988, 0.03806864, 1.16422331,
        0.65546478, 1.13456919, 0.22947233, 0.8434153 , 2.0318397 ,
        0.6177445 , 0.86743044, 2.56250592])]

In [62]:
dist_from_interest_point = []
for idx in range(4):
    points2d = image_list[frame_number - frame0][idx].project_with_proj_mat(gaussian_points_ew)[:,0:2]
    dist_from_interest_point.append(np.sqrt(np.sum((np.fliplr(points2d) - image_list[frame_number - frame0][idx].interest_points)**2,axis = 1)))
dist_from_interest_point

[array([0.63212019, 1.08992083, 0.23652125, 1.02173981, 1.02742255,
        0.93649677, 1.69207021, 0.82472999, 0.85993064, 0.44651407,
        0.4518457 , 1.35447819, 0.78713681, 0.64673155, 1.46948678,
        0.37495691, 0.4679009 , 2.06481273]),
 array([0.8031767 , 1.398691  , 0.82046776, 0.36525243, 1.15045387,
        0.67706577, 2.12387483, 1.10191409, 0.97923448, 0.23560342,
        0.90676991, 2.05163821, 1.44047865, 1.03461989, 0.91155657,
        0.62427519, 0.61598626, 1.35023623]),
 array([1.32757455, 0.68800885, 1.33940905, 1.3753937 , 1.14072991,
        1.26561264, 3.35090891, 0.88949789, 1.08276862, 0.81862826,
        1.25781423, 2.64753535, 1.11816553, 1.23868071, 2.06688308,
        1.64830059, 1.05302468, 1.71971231]),
 array([0.079269  , 0.90682305, 1.28828682, 1.8469423 , 0.68150755,
        0.43495377, 1.66792202, 0.93475347, 0.30327158, 0.51047383,
        0.53118798, 0.83525326, 0.89992593, 0.80143499, 2.24498325,
        0.95356646, 0.84154408, 2.39465877])]

In [59]:
points2d

array([[ 94.80730802,  84.29777965],
       [118.57972364,  88.55581411],
       [125.73197437,  96.96667001],
       [124.30309122, 101.1668775 ],
       [108.25384396, 105.51540965],
       [ 89.12091264,  91.84717873],
       [105.28511443,  95.02511612],
       [ 84.43229966,  79.65627264],
       [ 89.06293643,  54.01622465],
       [107.5328014 ,  37.49921321],
       [112.76389606,  35.69645474],
       [112.82007659,  39.09143015],
       [ 97.31154838,  54.67586689],
       [ 81.65016156,  64.07207735],
       [ 93.7209085 ,  51.84918444],
       [ 80.51170837,  68.82896714],
       [ 57.52361156,  94.72697655],
       [101.66395764,  64.85770762]])

In [ ]:

dist_from_interest_point

[array([21.6532813 , 66.16130425, 80.33306219, 79.42442593, 49.93410927,
        11.55457425, 42.28490987,  0.95792456,  7.04796386, 15.41273917,
        20.71433607, 19.81542778,  0.9389775 , 17.6787372 ,  1.72923168,
        14.19472641, 47.07225694, 24.68445189]),
 array([ 73.82524815, 103.3457253 , 103.11361707,  96.52885289,
         74.69012978,  55.46788706,  77.63213888,  44.70923831,
          1.82566064,  20.18045987,  37.05453662,  43.50785294,
         46.3999015 ,  11.40399893,  21.37735495,  16.4506747 ,
          4.01056565,  42.64645338]),
 array([ 7.98694841, 52.8609106 , 76.37246791, 83.2475674 , 64.40880232,
        10.54745613, 40.55544712, 15.061927  , 49.51479327, 46.64299533,
        41.08264414, 36.79749875, 35.93691202, 44.22265726, 43.41871055,
        39.77208573, 33.28827114, 10.29223355]),
 array([ 2.15511155, 24.84996676, 24.91928198, 16.69154874, 13.49971486,
        20.66889546,  1.49624688, 10.61285042, 32.30748397, 82.51187789,
        91.94967668, 86.

In [76]:
kps = 14
plt.imshow(image_list[450-370][0].im)
plt.scatter(image_list[450-370][0].interest_points[kps,1],image_list[450-370][0].interest_points[kps,0])